In [1]:
import rasterio
import numpy as np
import glob
import os
import csv
import random
import datetime
from tqdm import tqdm
from rasterio.warp import Resampling, reproject
import matplotlib.pyplot as plt
from pathlib import Path
from multiprocessing import Pool

In [ ]:
# -----------------------
#  USER CONFIG Jayden File Locations
# -----------------------
IMAGERY_ROOT = r"C:\Users\jaydb\Snowpack_Project\2019-2020_Data\Imagery"
MASK_ROOT    = r"C:\Users\jaydb\Snowpack_Project\Upscaled_Masks\Snow_Mask"

In [2]:
# -----------------------
#  USER CONFIG for Joel
# Directory paths for Joel's OneDrive setup, directly copied from "Initial_Analysis_and_Cleaning.ipynb"
ONEDRIVE_BASE = Path("/Users/joelcarlson/Library/CloudStorage/OneDrive-SharedLibraries-UCB-O365/Travis Hainsworth - RMBL")
SOURCE_FILEPATH = Path(ONEDRIVE_BASE / "2 - Source Data" / "2019-2020_Data")
IMAGERY_FOLDER = Path(SOURCE_FILEPATH / "Imagery")


In [ ]:

# -----------------------
#  HELPER FUNCTIONS
# -----------------------
def read_tiff_rasterio_keep_alpha(tiff_path, expected_nodata=None, ignore_nodata=False):
    """
    Reads *all* bands of a TIFF (including alpha, if present) into float32 (C, H, W).
    Replaces masked/nodata pixels with np.nan when not ignoring nodata.

    expected_nodata:
        A value to be treated as nodata and replaced with np.nan.
        For example, for masks pass -3.3999999521443642e+38.
        For imagery (if 0.0 is valid) pass None.
    ignore_nodata:
        If True, reads raw data without auto-masking (to preserve valid 0.0 pixels).
    """
    with rasterio.open(tiff_path) as src:
        bands_list = []
        for i in range(1, src.count + 1):
            if ignore_nodata:
                # Read raw data without auto-masking.
                band = src.read(i, masked=False)
            else:
                # Read as MaskedArray (this will mask nodata values)
                band = src.read(i, masked=True)
                band = band.filled(np.nan)
            band = band.astype(np.float32, copy=True)
            # If an expected nodata value is provided, replace it with np.nan.
            if expected_nodata is not None:
                band[band == expected_nodata] = np.nan
            bands_list.append(band)
        data = np.stack(bands_list, axis=0)  # (C, H, W)
        profile = src.profile
        profile["dtype"] = "float32"
    return data, profile

def tile_image(data, tile_size=512, overlap=0):
    """
    Splits a numpy array (C,H,W) (or (H,W)) into smaller patches.
    Returns a list of patches. Overlap is optional.
    """
    if data.ndim == 2:
        data = data[np.newaxis, :]  # make it (1, H, W)
    _, H, W = data.shape

    stride = tile_size - overlap
    patches = []

    for y in range(0, H, stride):
        for x in range(0, W, stride):
            y_end = min(y + tile_size, H)
            x_end = min(x + tile_size, W)
            patch = data[:, y:y_end, x:x_end]
            patches.append(patch)

    return patches

def visualize_random_pairs(image_mask_pairs, num_samples=3, normalize=True):
    """
    Randomly selects a few (image, mask) pairs and plots them side-by-side
    for a quick sanity check *before* tiling.
    """
    chosen = random.sample(image_mask_pairs, min(num_samples, len(image_mask_pairs)))

    for img_path, msk_path in chosen:
        # For images, disable auto-masking to preserve valid 0.0 pixels.
        img_data, _ = read_tiff_rasterio_keep_alpha(img_path, expected_nodata=None, ignore_nodata=True)
        # For masks, convert the extreme nodata value to nan.
        msk_data, _ = read_tiff_rasterio_keep_alpha(msk_path, expected_nodata=-3.3999999521443642e+38)

        print(f"\nPreviewing: {os.path.basename(img_path)} + {os.path.basename(msk_path)}")

        # Simple [0..1] normalization for the image if max > 1
        if normalize and np.nanmax(img_data) > 1.0:
            max_val = np.nanmax(img_data)
            img_data = img_data / (max_val if max_val != 0 else 1.0)

        # Convert (C,H,W) -> (H,W,C) if 3 or 4 channels
        if img_data.ndim == 3 and img_data.shape[0] in [3, 4]:
            img_disp = np.transpose(img_data[:3], (1,2,0))  # show only first 3 if 4 channels
        elif img_data.ndim == 3 and img_data.shape[0] == 1:
            img_disp = img_data[0]
        else:
            img_disp = img_data  # 2D or some other shape

        # Mask handling
        if msk_data.ndim == 3 and msk_data.shape[0] == 1:
            msk_disp = msk_data[0]
        else:
            msk_disp = msk_data

        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(img_disp, cmap='gray' if img_disp.ndim == 2 else None)
        axes[0].set_title("Snow Image")
        axes[0].axis('off')

        axes[1].imshow(msk_disp, cmap='gray')
        axes[1].set_title("Mask (0=NotSnow, 1=Snow)")
        axes[1].axis('off')

        plt.show()

def list_image_mask_pairs(imagery_root, mask_root,
                          image_suffix="_snow.tif",
                          mask_suffix="_snowbinary.tif"):
    """
    Recursively searches `imagery_root` for files ending in `*_snow.tif`.
    For each found image, tries to locate the matching mask in `mask_root`,
    based on the subfolder + base filename (with suffix replaced).

    Returns:
        A list of (image_path, mask_path) pairs that exist on disk.
    """
    image_paths = glob.glob(os.path.join(imagery_root, "**", f"*{image_suffix}"), recursive=True)

    pairs = []
    for img_path in image_paths:
        filename = os.path.basename(img_path)  # e.g. "ParadiseBasin_2019_07_10_snow.tif"
        location_subfolder = os.path.basename(os.path.dirname(img_path))  # e.g. "ParadiseBasin"
        mask_filename = filename.replace(image_suffix, mask_suffix)
        candidate_mask_path = os.path.join(mask_root, location_subfolder, mask_filename)

        if os.path.exists(candidate_mask_path):
            pairs.append((img_path, candidate_mask_path))
        else:
            # Uncomment the following line to see warnings for missing masks:
            # print(f"Warning: No mask found for {img_path}")
            pass

    return pairs

def load_and_tile_for_unet(
    imagery_root,
    mask_root,
    output_dir,
    tile_size=512,
    overlap=0,
    num_preview=3
):
    """
    1) Gathers (image, mask) pairs from IMAGERY_ROOT + MASK_ROOT.
    2) Optionally previews a few full-size pairs.
    3) Tiles each pair into tile_size x tile_size .npy patches (with overlap).
    4) Skips any tile that does not contain any 0s or 1s in the mask tile.
    5) Saves the tile pairs in output_dir as .npy.
    """
    # 1) Gather pairs
    pairs = list_image_mask_pairs(
        imagery_root,
        mask_root,
        image_suffix="_snow.tif",
        mask_suffix="_snowbinary.tif"
    )
    print(f"Found {len(pairs)} image/mask pairs.")

    # 2) Optional preview
    if pairs and num_preview > 0:
        visualize_random_pairs(pairs, num_samples=num_preview, normalize=True)

    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)
    total_tiles_saved = 0

    # 3) Process each pair
    for img_path, msk_path in pairs:
        base_name_img = os.path.splitext(os.path.basename(img_path))[0]
        # e.g. "ParadiseBasin_2019_07_10_snow"

        # Read image & mask:
        # For images, disable auto-masking (ignore_nodata=True) so that valid 0.0 pixels are kept.
        img_data, _ = read_tiff_rasterio_keep_alpha(img_path, expected_nodata=None, ignore_nodata=True)
        # For masks, convert the nodata value to np.nan.
        msk_data, _ = read_tiff_rasterio_keep_alpha(msk_path, expected_nodata=-3.3999999521443642e+38)

        # Normalize image to [0..1] if max > 1
        max_val = np.nanmax(img_data) or 1.0
        if max_val > 1.0:
            img_data = img_data / 255

        img_data = img_data.astype(np.float32)
        msk_data = msk_data.astype(np.float32)

        # Check shape compatibility
        if img_data.shape[-2:] != msk_data.shape[-2:]:
            print(f"Warning: mismatch shape -> {img_path} vs {msk_path}")
            continue

        # Tile the full images
        img_tiles = tile_image(img_data, tile_size=tile_size, overlap=overlap)
        msk_tiles = tile_image(msk_data, tile_size=tile_size, overlap=overlap)

        if len(img_tiles) != len(msk_tiles):
            print(f"Warning: tile count mismatch: {len(img_tiles)} vs {len(msk_tiles)}. Skipping {base_name_img}")
            continue

        # 4) Save each tile (skip tiles that do not contain any valid mask values (0 or 1))
        for i, (i_tile, m_tile) in enumerate(zip(img_tiles, msk_tiles)):
            # Check if the mask tile contains ONLY 0s or 1s
           unique_values = np.unique(m_tile)
           if not np.all(np.isin(unique_values, [0, 1])):
                continue  # Skip tiles with any value other than 0 or 1

           tile_basename = f"{base_name_img}__tile_{i:04d}"
           msk_tile_path = os.path.join(output_dir, tile_basename + "_msk.npy")
           img_tile_path = os.path.join(output_dir, tile_basename + "_img.npy")

           np.save(img_tile_path, i_tile)
           np.save(msk_tile_path, m_tile)
           total_tiles_saved += 1

        # Memory cleanup
        del img_data, msk_data, img_tiles, msk_tiles

    print(f"\nDone! Saved {total_tiles_saved} tile pairs in: {output_dir}")

In [ ]:
# -----------------------
#  RUN TILING
# -----------------------
tile_size = 512
overlap   = 64
out_dir   = r"C:\Users\jaydb\Snowpack_Project\Tiled_Output"
load_and_tile_for_unet(
     imagery_root=IMAGERY_ROOT,
     mask_root=MASK_ROOT,
     output_dir=out_dir,
     tile_size=tile_size,
     overlap=overlap,
     num_preview=0  # Number of random full-size pairs to preview before tiling
)

Found 131 image/mask pairs.
